# Generating Creative Text (Stories, Poems) Using Language Models

## 📚 Learning Objectives

By completing this notebook, you will:
- Train a language model on **real published stories** and generate new story text
- Train a second one on **real published poetry** and see how the corpus changes the output's *shape*, not just its words
- Control creativity with **temperature** and **top-k** sampling
- Evaluate creative outputs honestly — including what a small model cannot do

## 🔗 Where this fits

**Builds on:** Course 10 — Unit 2, lesson 01 "Text Generation with Autoregressive Models" — the same model, the same next-token loss minimised by gradient descent (Course 01 (AIAT 111) — Unit 3, lesson 04), with the training corpus as the only thing changed.

**Used later in:** Course 10 — Unit 2, lesson 06, which asks how such output can be scored at all.

## 📊 Data used in this notebook

Two **real** literary corpora from the NLTK Gutenberg collection — no invented sentences:
1. **Stories** — *Stories to Tell to Children* (Sara Cone Bryant) and *Alice's Adventures in Wonderland* (Lewis Carroll).
2. **Poetry** — *Poems* (William Blake, 1789), kept with its original line breaks so the model can learn verse layout.

Why this matters: a hand-written "storybook" sentence repeated ten times is memorised in a few hundred steps, so the "creative" output is just the corpus played back. On real books the model must actually learn style — and you can see the difference between prose and verse in the generated text.

---

This notebook covers practical activities from **Course 10, Unit 2**:
- Generating creative text (stories, poems) using language models

---

## Introduction

**Creative text generation** uses language models to produce stories, poems, and other creative content. The model is trained once; the *decoding strategy* — temperature and top-k — is what you turn to trade coherence against surprise.


## 📥 Inputs & 📤 Outputs

**Inputs:** real published prose and poetry from `nltk.corpus.gutenberg` (`bryant-stories.txt`, `carroll-alice.txt`, `blake-poems.txt`). If the corpus cannot be downloaded, the notebook falls back to real 20 Newsgroups posts — never to invented text.

**Outputs:** training losses for two character-level LSTMs, generated story text at three decoding settings (T=0.5, T=1.2, top-k=5), and generated verse from the poetry model with its line structure preserved.

---


In [1]:
# WHAT/WHY: train one character-LSTM on a REAL story corpus, then compare
# decoding strategies (temperature, top-k) on the SAME trained model — the
# sampler, not the model, is what changes the "creativity". Real books matter
# here: a repeated toy sentence would be memorised, and every "creative"
# sample would just be the corpus played back.
import torch, torch.nn as nn, torch.optim as optim, numpy as np
import re
print(f'PyTorch {torch.__version__}')
print('✅ Libraries imported!')
print('\nGenerating Creative Text: Stories and Poems')
print('=' * 60)
print('\nCreative Generation techniques demonstrated:')
print('  - Temperature sampling  (higher T → more random/creative)')
print('  - Top-k sampling        (only sample from top-k tokens)')
print('  - Character LSTM language model trained on REAL published stories')

torch.manual_seed(7); np.random.seed(7)

# ── REAL DATA: published children's stories + Alice in Wonderland ─────────
def load_stories():
    try:
        import nltk
        nltk.download('gutenberg', quiet=True)
        from nltk.corpus import gutenberg
        raw = gutenberg.raw('bryant-stories.txt') + ' ' + gutenberg.raw('carroll-alice.txt')
        return raw, "Bryant, 'Stories to Tell to Children' + Carroll, 'Alice in Wonderland'"
    except Exception:
        from sklearn.datasets import fetch_20newsgroups
        news = fetch_20newsgroups(subset='train', categories=['rec.sport.baseball'],
                                  remove=('headers', 'footers', 'quotes'))
        return " ".join(news.data), '20 Newsgroups posts (fallback)'

raw, story_source = load_stories()
corpus = re.sub(r'[^a-z .,]', ' ', raw.lower())
corpus = re.sub(r'\s+', ' ', corpus).strip()[:24000]
print(f'\nStory corpus: {story_source}')
print(f'  {len(corpus):,} characters | sample: {corpus[:130]!r}')

chars = sorted(set(corpus)); c2i = {c: i for i, c in enumerate(chars)}
i2c  = {i: c for c, i in c2i.items()}; V, S = len(chars), 40
print(f'  vocabulary: {V} characters | context window: {S}')

X = torch.tensor([[c2i[corpus[j+k]] for k in range(S)] for j in range(len(corpus)-S)], dtype=torch.long)
y = torch.tensor([c2i[corpus[j+S]] for j in range(len(corpus)-S)], dtype=torch.long)
print(f'  training windows: {len(X):,}')


# ── Model: embedding → LSTM → next-character scores ───────────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__(); self.e = nn.Embedding(V, 64); self.l = nn.LSTM(64, 256, batch_first=True); self.f = nn.Linear(256, V)
    def forward(self, x):
        return self.f(self.l(self.e(x))[0][:, -1, :])


# ── Training: 8 epochs of next-character prediction on real prose ─────────
model = CharLM(); opt = optim.Adam(model.parameters(), lr=3e-3); crit = nn.CrossEntropyLoss()
from torch.utils.data import TensorDataset, DataLoader
loader = DataLoader(TensorDataset(X, y), batch_size=256, shuffle=True)
for ep in range(6):
    model.train(); el = 0
    for xb, yb in loader:
        opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step(); el += loss.item()
    if (ep+1) % 2 == 0: print(f'Epoch {ep+1}: loss={el/len(loader):.4f}')


# ── Sampling: temperature rescales scores; top-k zeroes all but the k best ──
def generate(seed, n=120, temperature=1.0, top_k=0):
    model.eval(); out = seed; ctx = [c2i.get(c, 0) for c in seed[-S:]]
    ctx = ([0] * (S - len(ctx))) + ctx          # left-pad short seeds to S
    for _ in range(n):
        x = torch.tensor([ctx], dtype=torch.long)
        with torch.no_grad(): logits = model(x)[0] / max(temperature, 1e-6)
        if top_k > 0:
            vals, idx = torch.topk(logits, top_k); mask = torch.full_like(logits, -1e9); mask[idx] = vals; logits = mask
        probs = torch.softmax(logits, 0).numpy(); nxt = int(np.random.choice(V, p=probs))
        out += i2c[nxt]; ctx = ctx[1:] + [nxt]
    return out

np.random.seed(7)
print('\n--- Temperature 0.5 (conservative): ---')
print(generate('once upon a time there was a ', n=120, temperature=0.5))
print('\n--- Temperature 1.2 (creative): ---')
print(generate('once upon a time there was a ', n=120, temperature=1.2))
print('\n--- Top-k sampling (k=5): ---')
print(generate('the little girl looked at the ', n=120, temperature=1.0, top_k=5))
print('\nOne model, three samplers. Low temperature repeats the corpus\'s most')
print('frequent phrasing; high temperature invents non-words; top-k keeps the')
print('surprise but bans the worst 50-odd characters at each step.')


PyTorch 2.13.0
✅ Libraries imported!

Generating Creative Text: Stories and Poems

Creative Generation techniques demonstrated:
  - Temperature sampling  (higher T → more random/creative)
  - Top-k sampling        (only sample from top-k tokens)
  - Character LSTM language model trained on REAL published stories



Story corpus: Bryant, 'Stories to Tell to Children' + Carroll, 'Alice in Wonderland'
  24,000 characters | sample: 'stories to tell to children by sara cone bryant two little riddles in rhyme there s a garden that i ken, full of little gentlemen '
  vocabulary: 29 characters | context window: 40
  training windows: 23,960


Epoch 2: loss=1.6446


Epoch 4: loss=1.2459


Epoch 6: loss=1.0261

--- Temperature 0.5 (conservative): ---
once upon a time there was a corner and the sher lion in the gingerbread boy, said the little country mouse said the street one said the bed here and

--- Temperature 1.2 (creative): ---
once upon a time there was a come and, asleeps out all and antead wom the trukd ex said the cheery. i panter tapring of wheaver dear wheater the fox 

--- Top-k sampling (k=5): ---
the little girl looked at the sky. the lion and thinder whise his lion was even be at lasted take there, and the little old mone stoped. and anything 

One model, three samplers. Low temperature repeats the corpus's most
frequent phrasing; high temperature invents non-words; top-k keeps the
surprise but bans the worst 50-odd characters at each step.


## 🌍 Real-World Worked Example — A Poetry Model

**Industry context:**
- GitHub Copilot generates code token by token
- ChatGPT predicts the next token based on all previous context
- Autocomplete on your phone uses a smaller version of the same idea

Now change **only the corpus** and keep everything else identical: the same character-LSTM recipe, trained on **William Blake's *Poems* (1789)** — real published verse, with its line breaks kept in the text.

This is the experiment worth watching. Prose and verse differ in a way a character model can pick up: verse has short lines, so the newline character appears on a strong rhythm. If the model has genuinely learned the corpus's style rather than its words, the generated text should *look* like verse — broken into short lines — even where the words are nonsense.


In [2]:
# WHAT/WHY: identical model and training recipe, one change — the corpus is
# REAL published poetry instead of REAL published prose. Newlines are kept in
# the vocabulary so the model can learn line breaks, i.e. the shape of verse.
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import re

torch.manual_seed(42); np.random.seed(42)

# ── REAL DATA: William Blake, "Poems" (1789) ──────────────────────────────
def load_poems():
    try:
        import nltk
        nltk.download('gutenberg', quiet=True)
        from nltk.corpus import gutenberg
        return gutenberg.raw('blake-poems.txt'), 'Blake, "Poems" (1789), Gutenberg'
    except Exception:
        from sklearn.datasets import fetch_20newsgroups
        news = fetch_20newsgroups(subset='train', categories=['soc.religion.christian'],
                                  remove=('headers', 'footers', 'quotes'))
        return " ".join(news.data), '20 Newsgroups posts (fallback)'

raw, poem_source = load_poems()
# Keep "\n" this time: the line break is the character that carries verse shape.
text = re.sub(r'[^a-z .,\n]', ' ', raw.lower())
text = re.sub(r'[ \t]+', ' ', text)
text = re.sub(r'\n{3,}', '\n\n', text).strip()
print(f'Poetry corpus: {poem_source}')
print(f'  {len(text):,} characters, {text.count(chr(10)):,} line breaks')
print(f'  a real stanza from the corpus:\n{text[180:340]}')

# ── Vocabulary: map each character (newline included) to an integer id ────
chars  = sorted(set(text))
c2i    = {c: i for i, c in enumerate(chars)}
i2c    = {i: c for c, i in c2i.items()}
VOCAB  = len(chars)
enc    = [c2i[c] for c in text]
print(f'\n  vocabulary: {VOCAB} characters (newline is one of them)')

# ── Dataset: every 40-char window predicts the next character ─────────────
SEQ_LEN = 40
X_t = torch.tensor([enc[i:i+SEQ_LEN] for i in range(len(enc)-SEQ_LEN-1)], dtype=torch.long)
y_t = torch.tensor([enc[i+SEQ_LEN]   for i in range(len(enc)-SEQ_LEN-1)], dtype=torch.long)
print(f'  training windows: {len(X_t):,}')

# ── LSTM Language Model (same architecture as the story model) ────────────
class PoemLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 64)
        self.lstm  = nn.LSTM(64, 256, batch_first=True)
        self.fc    = nn.Linear(256, VOCAB)
    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.fc(out[:, -1, :])

pmodel  = PoemLM()
opt     = optim.Adam(pmodel.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

# ── Training: 1200 steps, a fresh random 256-window batch each time ───────
for step in range(900):
    pmodel.train()
    perm = torch.randperm(len(X_t))[:256]  # mini-batch
    loss = loss_fn(pmodel(X_t[perm]), y_t[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 300 == 0:
        print(f"Step {step} — loss: {loss.item():.3f}")
print(f"Step 899 — loss: {loss.item():.3f}")

# ── Generation with temperature sampling ──────────────────────────────────
def generate_poem(seed_str, steps=260, temperature=0.7):
    pmodel.eval()
    chars_out = list(seed_str)
    ctx = [c2i.get(c, 0) for c in seed_str[-SEQ_LEN:]]
    ctx = ([0] * (SEQ_LEN - len(ctx))) + ctx
    for _ in range(steps):
        inp = torch.tensor([ctx[-SEQ_LEN:]]).long()
        with torch.no_grad():
            logits = pmodel(inp)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        next_c = int(np.random.choice(len(probs), p=probs))
        chars_out.append(i2c[next_c])
        ctx.append(next_c)
    return ''.join(chars_out)

np.random.seed(1)
print("\n── Generated verse (temperature 0.7) ───────────────────────────")
print(generate_poem("the little lamb ", steps=260, temperature=0.7))
print("\n── Generated verse (temperature 1.1) ───────────────────────────")
print(generate_poem("and the night ", steps=200, temperature=1.1))
print("\nCompare the SHAPE with the story samples above: same architecture,")
print("same training recipe, different real corpus — and the output arrives in")
print("short verse lines instead of running prose. A language model copies the")
print("statistics of whatever text you feed it, layout included.")


Poetry corpus: Blake, "Poems" (1789), Gutenberg
  37,507 characters, 1,425 line breaks
  a real stanza from the corpus:
nt glee,
 on a cloud i saw a child,
 and he laughing said to me 
 
 pipe a song about a lamb 
 so i piped with merry cheer.
 piper, pipe that song again 
 so i 

  vocabulary: 30 characters (newline is one of them)


  training windows: 37,466
Step 0 — loss: 3.409


Step 300 — loss: 1.479


Step 600 — loss: 1.248


Step 899 — loss: 0.997

── Generated verse (temperature 0.7) ───────────────────────────
the little lamb hi can like from shany from his virgin sweet does 
and deceithed in the desert smile,
 when the vicion love, where smile a happy blosm.
 
 oh dan all my day,
 then calls the live, the evenion s sweet, and smile,
 and the happy pour s soft manience
 o return an

── Generated verse (temperature 1.1) ───────────────────────────
and the night over my healt, merripved thee doth and exse.
 
 oh viefly flow.
 
 mother s fountusobs of the sky. shat weve alack fork its desice me
and slow
 
 but doth in the age 
 grave thime man with dests rooe 

Compare the SHAPE with the story samples above: same architecture,
same training recipe, different real corpus — and the output arrives in
short verse lines instead of running prose. A language model copies the
statistics of whatever text you feed it, layout included.


## 📚 References & Further Reading

**Papers:**
- Radford et al. (2019) — [GPT-2: Language Models are Unsupervised Multitask Learners](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Brown et al. (2020) — [GPT-3](https://arxiv.org/abs/2005.14165)

**Interactive:**
- [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT) — build GPT in 300 lines
- [The Unreasonable Effectiveness of RNNs](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)

**State-of-the-Art:** GPT-4, Claude 3.5, Gemini 1.5 — all trained on trillions of tokens with transformer decoders.

## 📝 Summary

In **Generating Creative Text (Stories, Poems)** you trained the *same* character-level LSTM twice on two **real** literary corpora — published children's stories plus *Alice in Wonderland*, and William Blake's *Poems* — and changed nothing else.

What that experiment shows:
- **The sampler controls creativity, not the model.** On one fixed story model, T=0.5 produced repetitive, frequent phrasing, T=1.2 produced invented words, and top-k=5 kept the variety while banning the worst candidates at each step.
- **The corpus controls style, including layout.** The poetry model was given the newline character in its vocabulary and learned to emit short verse lines; the prose model, trained identically, produces running prose. Neither was told what a poem is.
- **Be honest about the ceiling.** A 256-unit LSTM trained on tens of thousands of characters produces text with the right *shape* and plausible word fragments, not meaning or rhyme. Real creative systems (GPT-4, Claude) get their coherence from ~5 orders of magnitude more data and parameters — the mechanism you ran here is the same one.
